# SQL로 조회 함수 만들기

SQLite에서 매개변수 SQL로 고객 주문을 조회합니다.

## 실행 환경 준비

`import sqlite3`는 Python 표준 라이브러리의 SQLite 기능을 불러옵니다. `TemporaryDirectory()`는 실습 뒤 정리할 임시 폴더 객체를, `sqlite3.connect(경로)`는 데이터베이스 연결 객체를 반환합니다. `row_factory = sqlite3.Row`는 조회 행을 열 이름으로 읽게 합니다. `execute("PRAGMA foreign_keys = ON")`은 이 연결에서 외래 키 검사를 켭니다.

In [1]:
# 임시 SQLite 파일과 행 이름 접근 방식을 준비합니다.
import sqlite3
from pathlib import Path
from tempfile import TemporaryDirectory

temp_dir = TemporaryDirectory()
database = Path(temp_dir.name) / "orders.db"
connection = sqlite3.connect(database)
connection.row_factory = sqlite3.Row
_ = connection.execute("PRAGMA foreign_keys = ON")

## 기본 실행

`execute()`는 SQL 문장 하나를 실행하고 `executescript()`는 세미콜론으로 구분된 여러 문장을 실행합니다. `commit()`은 준비한 테이블과 행을 파일에 확정하고 `close()`는 연결을 닫습니다. SQL은 `SELECT → FROM/JOIN → WHERE → ORDER BY → LIMIT` 순서로 작성하지만 데이터베이스는 개념적으로 원천과 연결을 정한 뒤 행을 거르고 반환 열과 순서를 정합니다.

In [2]:
# 기본 키와 외래 키 제약을 가진 두 테이블을 만듭니다.
connection.executescript("""
CREATE TABLE customers(customer_id TEXT PRIMARY KEY, name TEXT NOT NULL);
CREATE TABLE orders(order_id TEXT PRIMARY KEY, customer_id TEXT NOT NULL, status TEXT NOT NULL,
 amount INTEGER NOT NULL, FOREIGN KEY(customer_id) REFERENCES customers(customer_id));
INSERT INTO customers VALUES ('C001','김하나'),('C002','이두리');
INSERT INTO orders VALUES ('O101','C001','paid',32000),('O102','C002','pending',18000),('O103','C001','paid',27000);
""")
connection.commit()
connection.close()
print("준비된 주문: 3")

준비된 주문: 3


## 중간 결과 관찰

함수 입력은 데이터베이스 경로와 고객 번호입니다. `orders o`와 `customers c`는 긴 테이블 이름에 각각 `o`, `c`라는 짧은 별칭을 붙입니다. 그래서 `o.order_id`와 `c.name`처럼 열의 출처를 표시할 수 있습니다. `USING(customer_id)`은 두 테이블에서 이름이 같은 `customer_id` 열의 값이 같은 행을 연결합니다. `with db:`는 정상 종료 시 진행 중인 트랜잭션을 커밋하고 예외가 발생하면 롤백하지만 연결을 닫지는 않습니다. 그래서 `finally`에서 `close()`를 따로 호출합니다. 블록 안에서는 `JOIN`, `WHERE`, `ORDER BY`를 적용하고 `db.execute(sql, (customer_id,))`의 값 튜플을 물음표와 분리해 전달합니다. 반환값은 `sqlite3.Row`를 사전으로 변환한 목록입니다.

In [3]:
# 경로와 고객 번호를 받아 매개변수 SQL 결과를 사전 목록으로 반환합니다.
def find_customer_orders(database: str | Path, customer_id: str) -> list[dict]:
    sql = """SELECT o.order_id, c.name AS customer_name, o.status, o.amount
             FROM orders o JOIN customers c USING(customer_id)
             WHERE customer_id = ? ORDER BY amount DESC, order_id"""
    db = sqlite3.connect(database)
    try:
        db.row_factory = sqlite3.Row
        with db:
            return [dict(row) for row in db.execute(sql, (customer_id,))]
    finally:
        db.close()
rows = find_customer_orders(database, "C001")
print(rows)

[{'order_id': 'O101', 'customer_name': '김하나', 'status': 'paid', 'amount': 32000}, {'order_id': 'O103', 'customer_name': '김하나', 'status': 'paid', 'amount': 27000}]


## 값 하나를 바꿔 비교

SQL 문장과 처리 순서는 유지하고 고객 번호만 C001에서 C002로 바꿉니다. 반환 행이 한 건으로 줄어도 열 이름과 사전 목록이라는 반환 형식은 유지되는지 확인합니다.

In [4]:
# 고객 번호만 바꾸어 반환 행 수와 정렬 규칙을 확인합니다.
assert [row["order_id"] for row in rows] == ["O101", "O103"]
print(find_customer_orders(database, "C002"))
assert len(find_customer_orders(database, "C002")) == 1

[{'order_id': 'O102', 'customer_name': '이두리', 'status': 'pending', 'amount': 18000}]


## 실패 재현

입력 문자열을 SQL에 직접 붙이면 조건의 뜻이 바뀔 수 있습니다.

In [5]:
# 문자열 결합은 입력 일부를 SQL 조건식으로 해석하게 만듭니다.
unsafe_input = "C001' OR 1=1 --"
db = sqlite3.connect(database)
try:
    unsafe_sql = f"SELECT * FROM orders WHERE customer_id = '{unsafe_input}'"
    unsafe_rows = db.execute(unsafe_sql).fetchall()
finally:
    db.close()
print("문자열 결합 조회 행:", len(unsafe_rows))
assert len(unsafe_rows) == 3

문자열 결합 조회 행: 3


## 수정

SQL의 물음표 자리표시자와 값 튜플을 분리해 전달합니다.

In [6]:
# 매개변수 조회와 외래 키 강제를 각각 실제 오류·결과로 확인합니다.
assert find_customer_orders(database, unsafe_input) == []
db = sqlite3.connect(database)
try:
    db.execute("PRAGMA foreign_keys = ON")
    try:
        with db:
            db.execute("INSERT INTO orders VALUES ('O999','C999','paid',1000)")
    except sqlite3.IntegrityError as error:
        print(type(error).__name__, "부모 없는 주문 거부")
    else:
        raise AssertionError("외래 키가 강제되지 않았습니다.")
finally:
    db.close()
print("매개변수 조회 행: 0")
temp_dir.cleanup()

IntegrityError 부모 없는 주문 거부
매개변수 조회 행: 0
